In [ ]:
# This script is demonstrating practical implementation of the LazySlide Protocol - Slide Level Labels. It includes preprocessing steps as outlined here: 
#https://lazyslide.readthedocs.io/en/latest/tutorials/multiple_slides.html#

# To begin
    #1. source /mnt/data0/ikoluk/miniconda3/bin/activate
    #2. conda activate wsi_env2
    #3. cd /mnt/data0/ikoluk/lazyslide
    #4. jupyter lab --no-browser --port=8888

In [2]:
from pathlib import Path
import pooch, pandas as pd, numpy as np
from dask_cuda import LocalCUDACluster
from dask.distributed import Client, as_completed
from tqdm.auto import tqdm
import lazyslide as zs      
from wsidata import open_wsi 

In [3]:
DATA_DIR  = Path("/mnt/data0/ikoluk/wsi_project/data/GTEx_artery_dataset.csv.gz ")
CSV_NAME  = "GTEx_artery_dataset.csv.gz"
csv_path  = DATA_DIR / CSV_NAME

if not csv_path.exists():                      
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    pooch.retrieve(
        url="https://lazyslide.blob.core.windows.net/lazyslide-data/" + CSV_NAME,
        path=DATA_DIR, fname=CSV_NAME, known_hash=None,
    )

dataset = pd.read_csv(csv_path)

In [4]:
#Creates a dask cluster that uses GPU under port 8787 and gives a client object to run GPU tasks
#http://localhost:8787/status
from dask_cuda import LocalCUDACluster
from dask.distributed import Client

cluster = LocalCUDACluster(
    dashboard_address=":8787",
    protocol="tcp",
    resources={"GPU": 1},
)
client = Client(cluster)

In [ ]:
from huggingface_hub import login
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")
login(token=HF_TOKEN)

In [6]:
#Defines the fe process
import lazyslide as zs
import pooch, numpy as np

TERMS = ["calcification", "atherosclerosis"]

def wsi_feature_extraction(slide_id: str):
    from wsidata import open_wsi          # <- correct import, inside fn

    svs_path = pooch.retrieve(
        url=f"https://brd.nci.nih.gov/brd/imagedownload/{slide_id}",
        fname=f"{slide_id}.svs",
        path="gtex_artery_slides",
        known_hash=None,
    )

    wsi = open_wsi(svs_path)              # <- direct call
    zs.pp.find_tissues(wsi)
    zs.pp.tile_tissues(wsi, 256, mpp=0.5, background_fraction=0.5)

    zs.tl.feature_extraction(wsi, "uni2")
    zs.tl.feature_aggregation(wsi, "uni2")

    zs.tl.feature_extraction(wsi, "conch", pbar=False)
    embed = zs.tl.text_embedding(TERMS)
    zs.tl.text_image_similarity(wsi, embed, "conch")
    wsi.write()

    return zs.metrics.topk_scores(wsi["conch_tiles_text_similarity"], k=100)

In [7]:
#Launches all slide level jobs
futures = [
    client.submit(wsi_feature_extraction, sid, resources={"GPU": 1})
    for sid in dataset["Tissue Sample Id"]
]

from dask.distributed import as_completed
from tqdm.auto import tqdm
for _ in tqdm(as_completed(futures), total=len(futures)):
    pass

  0%|          | 0/45 [00:00<?, ?it/s]

2025-07-10 22:01:14,365 - distributed.worker - ERROR - Compute Failed
Key:       wsi_feature_extraction-7e2e67d2fae2c7aea29362e22fd847dd
State:     executing
Task:  <Task 'wsi_feature_extraction-7e2e67d2fae2c7aea29362e22fd847dd' wsi_feature_extraction(...)>
Exception: "ValueError('len(nodes) = 0, expected 1. Unable to read the NGFF file. Please report this bug and attach a minimal data example.')"
Traceback: '  File "/tmp/ipykernel_894959/790636288.py", line 17, in wsi_feature_extraction\n  File "/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/site-packages/wsidata/io/_wsi.py", line 112, in open_wsi\n    sdata = read_zarr(store)\n  File "/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/site-packages/spatialdata/_io/io_zarr.py", line 112, in read_zarr\n    element = _read_multiscale(f_elem_store, raster_type="image")\n  File "/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/site-packages/spatialdata/_io/io_raster.py", line 65, in _read_multiscale\n    raise V

Extracting features ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 01:1499% 00:01 00:03


/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
2025-07-10 22:26:13,015 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('wsi_feature_extraction-a700df42a6c6eca560ddb574879a882c')" coro=<Worker.execute() done, defined at /mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError


KeyboardInterrupt: 

Process Dask Worker process (from Nanny):
2025-07-10 22:26:15,054 - distributed.nanny - ERROR - Worker process died unexpectedly
Traceback (most recent call last):
  File "/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/site-packages/distributed/process.py", line 202, in _run
    target(*args, **kwargs)
  File "/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/site-packages/distributed/nanny.py", line 1023, in _run
    asyncio_run(run(), loop_factory=get_loop_factory())
  File "/mnt/data0/ikoluk/miniconda3/envs/wsi_env3/lib/python3.10/site-packages/distributed/compatibility.py", line 236, in asyncio_run
    return loop.run_until_complete(main)
  File "/mnt/data0/ikoluk/miniconda3/e

In [ ]:
#k sim scores
slide_scores = pd.DataFrame(
    np.vstack([f.result() for f in futures]),
    columns=TERMS,
    index=dataset["Tissue Sample Id"],
)
slide_scores.to_csv("slide_scores.csv")

In [ ]:
import seaborn as sns

sns.boxplot(
    data=slide_scores.join(dataset.set_index("Tissue Sample Id")),
    x="Pathology Categories",
    y="calcification",
)
client.shutdown()

In [ ]:
from wsidata import agg_wsi

dataset["slide"] = [f"gtex_artery_slides/{s}.svs" for s in dataset["Tissue Sample Id"]]
agg_data = agg_wsi(dataset, "uni2", wsi_col="slide", agg_key="agg_slide")
agg_data.obs = agg_data.obs.merge(slide_scores, on="Tissue Sample Id").set_index(
    "Tissue Sample Id"
)
agg_data
agg_data.write_h5ad("agg_uni2_features.h5ad")